In [1]:
import pandas as pd
import numpy as np

In [2]:
# Set random seed for reproducibility
np.random.seed(42)

In [3]:
# Number of samples
n_samples = 10000

In [4]:
# Simulate weather data
temperature = np.random.uniform(15, 35, n_samples)  # Temperature between 15°C and 35°C
humidity = np.random.uniform(40, 95, n_samples)     # Humidity between 40% and 95%
rainfall = np.random.uniform(0, 50, n_samples)      # Rainfall between 0mm and 50mm
wind_speed = np.random.uniform(0, 20, n_samples)    # Wind speed between 0km/h and 20km/h

In [5]:
# Simulate pest presence based on weather conditions
# Armyworms: Warm, humid conditions with moderate rainfall
armyworms = (
    (temperature > 25) & (humidity > 70) & (rainfall > 10) & (rainfall < 30)
).astype(int)

# Stalk Borers: Warm, dry conditions with low rainfall
stalk_borers = (
    (temperature > 28) & (humidity < 50) & (rainfall < 10)
).astype(int)

# Aphids: Moderate temperatures and high humidity
aphids = (
    (temperature > 20) & (temperature < 30) & (humidity > 60) & (rainfall > 5)
).astype(int)

In [6]:
# Combine into a DataFrame
data = pd.DataFrame({
    'Temperature': temperature,
    'Humidity': humidity,
    'Rainfall': rainfall,
    'Wind_Speed': wind_speed,
    'Armyworms': armyworms,
    'Stalk_Borers': stalk_borers,
    'Aphids': aphids
})

In [7]:
# Save dataset to CSV file
data.to_csv('maize_pest_data.csv', index=False)

In [8]:
print(data.head())

   Temperature   Humidity   Rainfall  Wind_Speed  Armyworms  Stalk_Borers  \
0    22.490802  60.550245  36.499916   12.762891          0             0   
1    34.014286  58.310165   9.225600    9.185849          0             0   
2    29.639879  49.688465  17.331985   19.289970          0             0   
3    26.973170  73.399667  33.164032    4.379569          0             0   
4    18.120373  66.214329  24.104467   11.757128          0             0   

   Aphids  
0       1  
1       0  
2       0  
3       1  
4       0  


In [12]:
#Imports to train model:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier

In [13]:
# Features and target variables
X = data[['Temperature', 'Humidity', 'Rainfall', 'Wind_Speed']]
y = data[['Armyworms', 'Stalk_Borers', 'Aphids']]

In [14]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
# Train a multi-label model
model = MultiOutputClassifier(RandomForestClassifier(random_state=42))
model.fit(X_train, y_train)

MultiOutputClassifier(estimator=RandomForestClassifier(random_state=42))

In [16]:
# Evaluate the model
y_pred = model.predict(X_test)
print("Multi-Label Model Evaluation:")
for i, pest in enumerate(['Armyworms', 'Stalk_Borers', 'Aphids']):
    print(f"{pest}:")
    print(classification_report(y_test.iloc[:, i], y_pred[:, i]))

Multi-Label Model Evaluation:
Armyworms:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1805
           1       1.00      1.00      1.00       195

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

Stalk_Borers:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1976
           1       1.00      0.96      0.98        24

    accuracy                           1.00      2000
   macro avg       1.00      0.98      0.99      2000
weighted avg       1.00      1.00      1.00      2000

Aphids:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1399
           1       1.00      1.00      1.00       601

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted av

In [19]:
#Save model:
import pickle
with open("maize_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved as maize_model.pkl")

Model saved as maize_model.pkl
